# Medical Insurance Charges Prediction
### Linear Regression + EDA | NumPy • Pandas • Matplotlib • Seaborn • scikit-learn

تنفيذ كامل لجميع مراحل المشروع باستخدام بيانات `insurance.csv` الأصلية من Kaggle.

In [ ]:
!pip -q install numpy pandas matplotlib seaborn scikit-learn

## 1. Import libraries + read the dataset
يحاول الدفتر قراءة `insurance.csv` محليًا أولًا، ثم تحميل نفس البيانات من نسخة عامة على GitHub إذا لم يكن الملف مرفوعًا.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/pplonski/datasets-for-start/refs/heads/master/insurance/insurance.csv"
data_source = "insurance.csv" if Path("insurance.csv").is_file() else DATA_URL
try:
    df = pd.read_csv(data_source)
except Exception as exc:
    raise RuntimeError("Download insurance.csv from Kaggle and upload it to Colab Files, then retry.") from exc
assert df.shape == (1338, 7), f"Unexpected dataset shape: {df.shape}"
print("Dataset size:", df.shape)
df.head()

## 2. EDA: head, info, describe, missing values

In [ ]:
display(df.head())
df.info()
display(df.describe(include="all"))
print("Missing values:")
print(df.isna().sum())

### Distribution of `charges`

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="charges", bins=35, kde=True, color="#246a91")
plt.title("Distribution of Medical Insurance Charges")
plt.xlabel("Charges (USD)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

### Pairplot: features vs target labels

In [ ]:
g = sns.pairplot(
    df[["age", "bmi", "children", "charges", "smoker"]],
    x_vars=["age", "bmi", "children"],
    y_vars=["charges"],
    hue="smoker",
    height=3.2,
    plot_kws={"alpha": 0.5, "s": 17},
)
g.figure.suptitle("Features vs. Charges by Smoking Status", y=1.06)
plt.show()

### Label encoding + correlation heatmap

In [ ]:
df_encoded = df.copy()
encoders = {}
for column in ["sex", "smoker", "region"]:
    encoder = LabelEncoder()
    df_encoded[column] = encoder.fit_transform(df_encoded[column])
    encoders[column] = encoder
    print(column, dict(enumerate(encoder.classes_)))

display(df_encoded.head())
plt.figure(figsize=(9, 6))
sns.heatmap(df_encoded.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between Features and Charges")
plt.tight_layout()
plt.show()

## 3. Training & Testing Split
70% تدريب، 30% اختبار، وقيمة `random_state=23`.

In [ ]:
X = df_encoded.drop(columns="charges")
y = df_encoded["charges"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.70, test_size=0.30, random_state=23
)
print("Training data:", X_train.shape, y_train.shape)
print("Testing data:", X_test.shape, y_test.shape)

## 4. Train Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Training complete.")

## 5. Test Model: intercept, coefficients, predictions, actual values

In [ ]:
predictions = model.predict(X_test)
real_values = y_test.to_numpy()
print("Intercept:", model.intercept_)
display(pd.DataFrame({"Feature": X.columns, "Coefficient": model.coef_}))
print("Predictions:", np.round(predictions, 2))
print("Real_Values:", np.round(real_values, 2))
comparison = pd.DataFrame({
    "Real_Values": real_values,
    "Predictions": predictions,
    "Residuals": real_values - predictions,
})
display(comparison.head(20))

### Real Values vs Predictions (Scatter Plot)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=real_values, y=predictions, alpha=0.6)
low = min(real_values.min(), predictions.min())
high = max(real_values.max(), predictions.max())
plt.plot([low, high], [low, high], "--", color="firebrick", label="Perfect prediction")
plt.title("Actual vs Predicted Charges")
plt.xlabel("Real Values (USD)")
plt.ylabel("Predictions (USD)")
plt.legend()
plt.tight_layout()
plt.show()

### Residual Histogram

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(comparison["Residuals"], bins=30, kde=True, color="#246a91")
plt.axvline(0, color="firebrick", linestyle="--")
plt.title("Residuals: Real Values - Predictions")
plt.xlabel("Residual (USD)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 6. Evaluate: MAE, MSE, RMSE, R²

In [ ]:
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)
metrics = pd.DataFrame({"Metric": ["MAE", "MSE", "RMSE", "R2"], "Value": [mae, mse, rmse, r2]})
display(metrics)
comparison.to_csv("predictions_vs_real_values.csv", index=False)
metrics.to_csv("insurance_model_metrics.csv", index=False)
print("Saved results as CSV files.")

### Understanding the result
- **MAE / MSE / RMSE:** الأقل أفضل.
- **R²:** الأقرب إلى 1 أفضل؛ هنا المتوقع نحو 0.7289.
- **الملاحظة:** ترميز `region` باستخدام `LabelEncoder` قد يضيف ترتيبًا عدديًا غير حقيقي. استخدمناه امتثالًا لمتطلبات الدورة؛ عمليًا يفضّل `OneHotEncoder` للمتغيرات الاسمية.